
# 02 - AcmeNet Documents to Chunks

Goal:
Convert raw AcmeNet support documents from the Bronze Delta table into searchable text chunks.

Input:
acmenet_bronze_documents

Output:
acmenet_silver_chunks

In [0]:
bronze_df = spark.table("acmenet_bronze_documents")

display(
    bronze_df.select(
        "document_name",
        "section",
        "source",
        "content_length",
        "loaded_at"
    )
)

document_name,section,source,content_length,loaded_at
troubleshooting_guide.md,Slow Internet,troubleshooting,187,2026-05-23T05:50:11.972Z
billing_policy.md,Billing Disputes,billing,158,2026-05-23T05:50:11.972Z
refund_policy.md,Refund Eligibility,refunds,154,2026-05-23T05:50:11.972Z


In [0]:
document_count = bronze_df.count()

print(f"Documents loaded from Bronze table: {document_count}")

if document_count == 0:
    raise Exception("Bronze table is empty. Please run Notebook 01 first")

Documents loaded from Bronze table: 3


In [0]:
display(
    bronze_df.select(
        "document_name",
        "section",
        "content"
    )
)

document_name,section,content
troubleshooting_guide.md,Slow Internet,"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support."
billing_policy.md,Billing Disputes,Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.
refund_policy.md,Refund Eligibility,Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.


In [0]:
def chunk_text_by_words(text: str, chunk_size: int = 35, overlap: int = 8) -> list[str]:
    words = text.split()

    if not words:
        return []
    
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap
    return chunks


In [0]:
sample_text = "Customers experiencing slow internet should restart the modem before contacting support. If the problem continues, the case may require escalation."

sample_chunks = chunk_text_by_words(sample_text, chunk_size=10, overlap=3)

for index, chunk in enumerate(sample_chunks, start=1):
    print(f"Chunk {index}: {chunk}")



Chunk 1: Customers experiencing slow internet should restart the modem before contacting
Chunk 2: modem before contacting support. If the problem continues, the case
Chunk 3: continues, the case may require escalation.


In [0]:
#Convert documents in chunks
import re

bronze_rows = bronze_df.select(
    "document_name",
    "section",
    "source",
    "content"
).collect()

chunk_rows = []

for row in bronze_rows:
    document_name = row["document_name"]
    section = row["section"]
    source = row["source"]
    content = row["content"]

    document_key = re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        document_name.replace(".md", "").lower()
    ).strip("_")

    chunks = chunk_text_by_words(
        text=content,
        chunk_size=35,
        overlap=8
    )

    for chunk_index, chunk_text in enumerate(chunks, start=1):
        chunk_rows.append({
            "chunk_id": f"{document_key}_{chunk_index:03d}",
            "document_name": document_name,
            "section": section,
            "source": source,
            "chunk_index": chunk_index,
            "chunk_text": chunk_text,
            "chunk_word_count": len(chunk_text.split())
        })

print(f"Generated chunks: {len(chunk_rows)}")


Generated chunks: 3


In [0]:
chunks_df = spark.createDataFrame(chunk_rows)

display(chunks_df)
#)

chunk_id,chunk_index,chunk_text,chunk_word_count,document_name,section,source
troubleshooting_guide_001,1,"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.",30,troubleshooting_guide.md,Slow Internet,troubleshooting
billing_policy_001,1,Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.,25,billing_policy.md,Billing Disputes,billing
refund_policy_001,1,Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.,26,refund_policy.md,Refund Eligibility,refunds


In [0]:
from pyspark.sql.functions import current_timestamp, length, col

silver_df = (
    chunks_df
    .withColumn("chunk_char_count", length(col("chunk_text")))
    .withColumn("created_at", current_timestamp())
)

display(silver_df)


chunk_id,chunk_index,chunk_text,chunk_word_count,document_name,section,source,chunk_char_count,created_at
troubleshooting_guide_001,1,"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.",30,troubleshooting_guide.md,Slow Internet,troubleshooting,187,2026-05-23T06:48:23.639Z
billing_policy_001,1,Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.,25,billing_policy.md,Billing Disputes,billing,158,2026-05-23T06:48:23.639Z
refund_policy_001,1,Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.,26,refund_policy.md,Refund Eligibility,refunds,154,2026-05-23T06:48:23.639Z


In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("acmenet_silver_chunks")

In [0]:
silver_table = spark.table("acmenet_silver_chunks")

display(
    silver_table.select(
        "chunk_id",
        "document_name",
        "source",
        "chunk_index",
        "chunk_text"
    )
)

chunk_id,document_name,source,chunk_index,chunk_text
troubleshooting_guide_001,troubleshooting_guide.md,troubleshooting,1,"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support."
billing_policy_001,billing_policy.md,billing,1,Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.
refund_policy_001,refund_policy.md,refunds,1,Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.


In [0]:
%sql

SELECT
  document_name,
  COUNT(*) AS total_chunks,
  AVG(chunk_word_count) AS avg_words_per_chunk,
  AVG(chunk_char_count) AS avg_chars_per_chunk
FROM acmenet_silver_chunks
GROUP BY document_name
ORDER BY total_chunks DESC;

document_name,total_chunks,avg_words_per_chunk,avg_chars_per_chunk
troubleshooting_guide.md,1,30.0,187.0
billing_policy.md,1,25.0,158.0
refund_policy.md,1,26.0,154.0


In [0]:
%sql

SELECT
  source,
  chunk_index,
  chunk_text
FROM acmenet_silver_chunks
ORDER BY source, chunk_index;

source,chunk_index,chunk_text
billing,1,Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.
refunds,1,Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.
troubleshooting,1,"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support."


In [0]:
%sql
SELECT
  document_name,
  COUNT(*) AS total_chunks,
  AVG(chunk_word_count) AS avg_words_per_chunk,
  AVG(chunk_char_count) AS avg_chars_per_chunk
FROM acmenet_silver_chunks
GROUP BY document_name
ORDER BY total_chunks DESC;

document_name,total_chunks,avg_words_per_chunk,avg_chars_per_chunk
troubleshooting_guide.md,1,30.0,187.0
billing_policy.md,1,25.0,158.0
refund_policy.md,1,26.0,154.0
